In [1]:
from pathlib import Path 
import os
import dotenv
dotenv.load_dotenv()
os.chdir(Path(os.getenv("PYTHONPATH")).expanduser())

In [2]:
import pandas as pd
import time
import numpy as np
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from skorch import NeuralNetRegressor
import torch
import torch.nn as nn
from src.model.ESN import ESNRegressor

In [3]:
df = pd.read_pickle('data/lorenz_data.pkl')
df.drop(columns=["dt_min", "t_max"], inplace=True)
df.head()

,t,x,y,z
0,0.000,-7.906697,-14.129764,-16.306456
1,0.001,-7.968928,-14.465952,-16.151253
2,0.002,-8.033898,-14.803325,-15.992904
3,0.003,-8.101592,-15.141956,-15.831328
4,0.004,-8.171996,-15.481917,-15.666437


In [4]:
STRIDE = 100
df_strides = df[ ::STRIDE ].reset_index(drop=True)

In [5]:
df_strides.head()

,t,x,y,z
0,0.0,-7.906697,-14.129764,-16.306456
1,0.1,-24.672225,-42.914606,35.172661
2,0.2,-10.285805,23.156058,48.685025
3,0.3,10.837283,18.620737,42.387750
4,0.4,8.634182,0.248069,41.103149


In [6]:
df_strides[100:105]

,t,x,y,z
100,10.0,5.621907,7.848892,21.957250
101,10.1,8.172017,11.064759,22.585364
102,10.2,10.855023,12.750135,27.747833
103,10.3,10.888821,8.897440,32.231589
104,10.4,7.908417,4.722127,30.117574


In [7]:
df_strides = df_strides[100:].reset_index(drop=True)
df_strides.head()

,t,x,y,z
0,10.0,5.621907,7.848892,21.957250
1,10.1,8.172017,11.064759,22.585364
2,10.2,10.855023,12.750135,27.747833
3,10.3,10.888821,8.897440,32.231589
4,10.4,7.908417,4.722127,30.117574


In [8]:
rng = np.random.default_rng(0)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
data = df_strides.to_numpy(dtype=np.float32)


In [ ]:
from src.data import LorenzWindowViewDataset, ToSkorchXY
from src.util import build_windows, split_indices
from torch.utils.data import DataLoader

series = df_strides[["x", "y", "z"]].to_numpy(dtype=np.float32) 
dev_series = series[:10000]    # (T, 3)
D = dev_series.shape[1]        # 3
WARM, XLEN = 10, 100
windows = build_windows(dev_series, warm_length=WARM, x_length=XLEN)   # (N, W+X+1, 3) view
N = windows.shape[0]

# --- split into *indices* (not materialized arrays) ---
train_idx, val_idx = split_indices(N, train_ratio=0.8, seed=42)

mu  = windows[train_idx].reshape(-1, D).mean(axis=0).astype(np.float32)
sd  = windows[train_idx].reshape(-1, D).std(axis=0).astype(np.float32)
sd  = np.where(sd == 0.0, 1.0, sd)  # guard against zeros

train_ds = LorenzWindowViewDataset(windows, train_idx, WARM, XLEN, standardize=True, mean=mu, std=sd)
val_ds  = LorenzWindowViewDataset(windows, val_idx,  WARM, XLEN, standardize=True, mean=mu, std=sd)


train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0, pin_memory=False)
val_loader  = DataLoader(val_ds,  batch_size=4, shuffle=False, num_workers=0, pin_memory=False)


train_ds = ToSkorchXY(LorenzWindowViewDataset(
    window_view=windows, indices=train_idx, warm_length=WARM, x_length=XLEN,
    standardize=False
))
val_ds = ToSkorchXY(LorenzWindowViewDataset(
    window_view=windows, indices=val_idx, warm_length=WARM, x_length=XLEN,
    standardize=False
))


net = NeuralNetRegressor(
    ESNRegressor,
    module__input_size=3,
    module__output_size=3,
    module__reservoir_size=500,
    module__leak=0.7,
    module__spectral_radius=0.9,
    module__input_scale=0.8,
    module__bias_scale=0.0,
    module__sparsity=0.05,
    module__activation='tanh',
    module__seed=42,
    criterion=nn.MSELoss,
    optimizer=torch.optim.Adam,
    optimizer__lr=2e-3,
    max_epochs=12,
    batch_size=256,
    train_split=None,              # no internal val split
    iterator_train__shuffle=False, # keep time order
    device=device,
    verbose=0,
)

# Train with skorch
#net.fit(train_ds, y=None)

,module,<class 'src.m...ESNRegressor'>
,criterion,<class 'torch...loss.MSELoss'>
,optimizer,<class 'torch...im.adam.Adam'>
,lr,0.01
,max_epochs,12
,batch_size,256
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>
,train_split,None
,callbacks,None


In [ ]:
# --- PREREQS (already built earlier) ------------------------------------------
# series:         np.ndarray (T, D) in ORIGINAL scale (float32)
# wins:           sliding_window_view(series, TOTAL=WARM+XLEN+1, axis=0)  -> (N, TOTAL, D)
# train_idx, val_idx: index arrays into wins (contiguous split recommended)
# mu, sd:         per-feature train-only mean/std (float32), shape (D,)
# WARM, XLEN:     warm and x lengths (e.g., 10, 100)

import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from skorch import NeuralNetRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

def ds_to_arrays(ds):
    Xs, Ys = [], []
    for i in range(len(ds)):
        X_i, y_i = ds[i]
        Xs.append(X_i.numpy())
        Ys.append(y_i.numpy())
    return np.stack(Xs, 0), np.stack(Ys, 0)

# --- Search space (trim or extend based on your ESNRegressor) -----------------
param_distributions = {
    # reservoir dynamics
    'module__reservoir_size': [300, 500, 700, 1000],
    'module__leak': [0.5, 0.7, 0.9],
    'module__spectral_radius': [0.7, 0.85, 1.0],
    'module__input_scale': [0.4, 0.6, 0.9, 1.2],
    'module__sparsity': [0.02, 0.03, 0.05],
    # include only if supported by your ESNRegressor:
    # 'module__bias_scale': [0.0, 0.1, 0.2],
    # 'module__activation': ['tanh', 'relu'],
    # 'module__seed': [42, 123, 314],

    # optimization
    'optimizer__lr': [1e-3, 2e-3, 3e-3],
    'optimizer__weight_decay': [1e-4, 1e-3],
    'batch_size': [16,32, 64],
    'max_epochs': [12, 25],
}


cv = TimeSeriesSplit(n_splits=3)

gs = RandomizedSearchCV(
    net,
    param_distributions=param_distributions,
    n_iter=100,                          # cut if you want faster search
    cv=cv,
    scoring='neg_mean_squared_error',   # standardized target space
    refit=True,
    n_jobs=-1,
    verbose=1,
    random_state=0
)

# standardize ToConcatWarmWindowView outputs on the fly
def ds_to_arrays_std(ds, mu, sd):
    Xs, Ys = [], []
    for i in range(len(ds)):
        X_i, y_i = ds[i]             # original scale tensors
        X_i = (X_i.numpy() - mu) / sd
        y_i = (y_i.numpy() - mu) / sd
        Xs.append(X_i); Ys.append(y_i)
    return np.stack(Xs, 0), np.stack(Ys, 0)

X_train, y_train = ds_to_arrays_std(train_ds, mu, sd)
X_val,   y_val   = ds_to_arrays_std(val_ds,   mu, sd)

# fit search exactly as you have it...
gs.fit(X_train, y_train)

# original-space metrics
from sklearn.metrics import mean_squared_error
def inv(z): return z * sd + mu

# rollout in std space, invert once
def rollout_std(est, start_window_orig, steps):
    cur = (start_window_orig - mu) / sd
    preds = []
    for _ in range(steps):
        y_next = est.predict(cur[None, ...]).astype(np.float32)[0]
        preds.append(y_next)
        cur = np.concatenate([cur[1:], y_next[None, :]], axis=0)
    return inv(np.stack(preds, 0))


Fitting 3 folds for each of 100 candidates, totalling 300 fits
Val MSE (orig): 0.0723760724067688
10-step rollout MSE (orig): 96.53128051757812


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Return STANDARDIZED predictions
def rollout_std(est, start_window_orig, steps):
    cur = (start_window_orig - mu) / sd
    preds_std = []
    for _ in range(steps):
        y_next_std = est.predict(cur[None, ...]).astype(np.float32)[0]
        preds_std.append(y_next_std)
        cur = np.concatenate([cur[1:], y_next_std[None, :]], axis=0)
    return np.stack(preds_std, axis=0)  # <-- standardized

# 10-step rollout (STANDARDIZED)
abs_i0 = int(val_idx[0])
start_window_orig = windows[abs_i0, :WARM+XLEN, :]
K = 10
preds_std = rollout_std(gs, start_window_orig, K)
gt_std = (dev_series[abs_i0 + WARM + XLEN : abs_i0 + WARM + XLEN + K] - mu) / sd


print(f"{K}-step rollout MSE (std):", mean_squared_error(gt_std, preds_std))
print("per-dim MSE:", mean_squared_error(gt_std, preds_std, multioutput="raw_values"))
print("R² per dim:", r2_score(gt_std, preds_std, multioutput="raw_values"))


10-step rollout MSE (std): 1.2970025539398193
per-dim MSE: [0.6939823 1.1675884 2.029437 ]
R² per dim: [-0.11108804 -1.3986189  -1.1254017 ]


In [36]:
import pandas as pd

cv = pd.DataFrame(gs.cv_results_)
cols = ["rank_test_score", "mean_test_score", "std_test_score", "params"]
topk = (cv[cols]
        .sort_values("rank_test_score")
        .assign(mean_test_MSE=lambda d: -d["mean_test_score"])  # flip sign if using neg MSE
        .head(10))

print(topk[["rank_test_score","mean_test_MSE","std_test_score","params"]]
      .to_string(index=False))


 rank_test_score  mean_test_MSE  std_test_score                                                                                                                                                                                                                                      params
               1       0.000820        0.000360   {'optimizer__weight_decay': 0.0001, 'optimizer__lr': 0.002, 'module__spectral_radius': 1.0, 'module__sparsity': 0.05, 'module__reservoir_size': 700, 'module__leak': 0.9, 'module__input_scale': 0.9, 'max_epochs': 25, 'batch_size': 64}
               2       0.001169        0.000393   {'optimizer__weight_decay': 0.001, 'optimizer__lr': 0.002, 'module__spectral_radius': 0.7, 'module__sparsity': 0.05, 'module__reservoir_size': 1000, 'module__leak': 0.9, 'module__input_scale': 1.2, 'max_epochs': 25, 'batch_size': 64}
               3       0.001224        0.000381   {'optimizer__weight_decay': 0.001, 'optimizer__lr': 0.002, 'module__spectral_radius': 1.0, 'module